#  ⭐ `fat_reacoes`


In [36]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, mapping_column, normalize_date_column, normalize_rows 

In [37]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [38]:
path = Path(project_root) / "data/01_bronze/Reacoes/Reacoes.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT',
       'HLT', 'HLGT', 'SOC', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO'],
      dtype='object')

### Quality

In [39]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,REACAO_EVTO_ADVERSO_MEDDRA_LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17


### GRAVE

In [40]:
bronze.GRAVE.value_counts()

GRAVE
Não    374682
Sim    268114
Name: count, dtype: int64

### DESFECHO

In [41]:
bronze.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         278670
Desconhecido                                 226494
Não Recuperado/Não Resolvido/Em andamento     85241
Em recuperação/Resolvendo                     66908
Fatal/Óbito                                   18581
Recuperado/Resolvido com sequelas              4311
Name: count, dtype: int64

### GRAVIDADE

In [42]:
bronze.GRAVIDADE.value_counts()

GRAVIDADE
Outro efeito clinicamente significativo                                                                                                                                                                              144833
Hospitalização/Prolongamento de hospitalização                                                                                                                                                                        57343
Hospitalização/Prolongamento de hospitalização, Outro efeito clinicamente significativo                                                                                                                               15621
Resultou em óbito                                                                                                                                                                                                     11510
Ameaça à vida                                                                                                 

### DURACAO

In [43]:
resume = bronze.DURACAO.value_counts().reset_index()
resume.head(10)

,DURACAO,count
0,1 dia,27059
1,2 dia,10017
2,3 dia,6650
3,1 hora,6314
4,30 minuto,5865
5,4 dia,4278
6,5 dia,3707
7,0 dia,3349
8,2 hora,3112
9,20 minuto,2698


# 🥈 Silver

normalized data, medium quality


In [44]:
silver = bronze.copy()

In [45]:
dim_soc = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc/dim_soc.parquet")
dim_soc.head()

,PK_SOC,SOC,HLGT,HLT,PT,PK_LLT,REACAO_EVTO_ADVERSO_MEDDRA_LLT
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual
2,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,3,Atividade sexual aumentada
3,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,4,Não consumação
4,26,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,5,Neonato


### pk

In [46]:
cols = ['IDENTIFICACAO_NOTIFICACAO', 'PK_SOC', 'PK_LLT','DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO', 'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO']
hist_fat_reacoes = silver.merge(dim_soc, on=['REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT', 'HLT', 'HLGT','SOC'], how='left')
hist_fat_reacoes = hist_fat_reacoes[cols]
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,16,4114,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,9,9107,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
2,BR-ANVISA-300000007,16,3903,20181115,None,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
3,BR-ANVISA-300000008,12,11411,20181025,None,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
4,BR-ANVISA-300000010,8,1994,201508,201508,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17


#### date

In [47]:

col_dates = ["DATA_INICIO_HORA", "DATA_FINAL_HORA"]

for col in col_dates:
    if col in hist_fat_reacoes.columns:
        hist_fat_reacoes[col] = normalize_date_column(hist_fat_reacoes[col])

hist_fat_reacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815877 entries, 0 to 815876
Data columns (total 10 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO  815877 non-null  object        
 1   PK_SOC                     770393 non-null  Int64         
 2   PK_LLT                     815877 non-null  int64         
 3   DATA_INICIO_HORA           347696 non-null  datetime64[ns]
 4   DATA_FINAL_HORA            189475 non-null  datetime64[ns]
 5   DURACAO                    127726 non-null  object        
 6   GRAVE                      642796 non-null  object        
 7   GRAVIDADE                  267804 non-null  object        
 8   DESFECHO                   680205 non-null  object        
 9   ATUALIZADO                 815877 non-null  object        
dtypes: Int64(1), datetime64[ns](2), int64(1), object(6)
memory usage: 63.0+ MB


In [48]:
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,16,4114,NaT,NaT,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,9,9107,2018-11-22,2018-11-22,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
2,BR-ANVISA-300000007,16,3903,2018-11-15,NaT,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
3,BR-ANVISA-300000008,12,11411,2018-10-25,NaT,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
4,BR-ANVISA-300000010,8,1994,NaT,NaT,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17


### GRAVE

In [49]:
hist_fat_reacoes.GRAVE.value_counts()

GRAVE
Não    374682
Sim    268114
Name: count, dtype: int64

In [50]:
 
grave_map = {
    "Sim": 1,
    "Não": 2, 
}
hist_fat_reacoes['GRAVE'] = hist_fat_reacoes['GRAVE'].map(grave_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes[cols]
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,16,4114,NaT,NaT,3 dia,2,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,9,9107,2018-11-22,2018-11-22,None,1,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
2,BR-ANVISA-300000007,16,3903,2018-11-15,NaT,2 dia,1,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
3,BR-ANVISA-300000008,12,11411,2018-10-25,NaT,5 dia,1,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
4,BR-ANVISA-300000010,8,1994,NaT,NaT,None,1,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17


In [51]:
hist_fat_reacoes.GRAVE.value_counts()

GRAVE
2    374682
1    268114
0    173081
Name: count, dtype: Int64

###  DESFECHO

In [53]:
hist_fat_reacoes.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         278670
Desconhecido                                 226494
Não Recuperado/Não Resolvido/Em andamento     85241
Em recuperação/Resolvendo                     66908
Fatal/Óbito                                   18581
Recuperado/Resolvido com sequelas              4311
Name: count, dtype: int64

In [54]:
desfecho_map = {
    "Desconhecido": 0, 
    "Não Recuperado/Não Resolvido/Em andamento": 1, 
    "Em recuperação/Resolvendo": 2, 
    "Recuperado/Resolvido": 3,
    "Fatal/Óbito": 4, 
    "Recuperado/Resolvido com sequelas": 5, 
}
hist_fat_reacoes['DESFECHO'] = hist_fat_reacoes['DESFECHO'].map(desfecho_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes[cols]
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,PK_SOC,PK_LLT,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,16,4114,NaT,NaT,3 dia,2,None,3,2025-11-17
1,BR-ANVISA-300000005,9,9107,2018-11-22,2018-11-22,None,1,Outro efeito clinicamente significativo,3,2025-11-17
2,BR-ANVISA-300000007,16,3903,2018-11-15,NaT,2 dia,1,Outro efeito clinicamente significativo,3,2025-11-17
3,BR-ANVISA-300000008,12,11411,2018-10-25,NaT,5 dia,1,Outro efeito clinicamente significativo,3,2025-11-17
4,BR-ANVISA-300000010,8,1994,NaT,NaT,None,1,Hospitalização/Prolongamento de hospitalização,3,2025-11-17


### GRAVIDADE

In [55]:
hist_fat_reacoes.to_parquet(Path(project_root) / "data/02_silver/hist_fat_reacoes/hist_fat_reacoes.parquet", index=False)

# 🥇 Gold


In [56]:
hist_fat_reacoes.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'PK_SOC', 'PK_LLT', 'DATA_INICIO_HORA',
       'DATA_FINAL_HORA', 'DURACAO', 'GRAVE', 'GRAVIDADE', 'DESFECHO',
       'ATUALIZADO'],
      dtype='object')

In [57]:
fat_reacoes = hist_fat_reacoes[['IDENTIFICACAO_NOTIFICACAO', 'PK_SOC', 'PK_LLT', 'DATA_INICIO_HORA',
       'DATA_FINAL_HORA', 'DURACAO', 'GRAVE', 'GRAVIDADE', 'DESFECHO']].copy()
fat_reacoes.to_parquet(Path(project_root) / "data/03_gold/fat_reacoes/fat_reacoes.parquet", index=False)